#  Sequence-to-Sequence Learning for Machine Translation
:label:`sec_seq2seq`

In so-called sequence-to-sequence problems such as machine translation
(as discussed in :numref:`sec_machine_translation`),
where inputs and outputs each consist 
of variable-length unaligned sequences,
we generally rely on encoder--decoder architectures
(:numref:`sec_encoder-decoder`).
In this section,
we will demonstrate the application 
of an encoder--decoder architecture,
where both the encoder and decoder 
are implemented as RNNs,
to the task of machine translation
:cite:`Sutskever.Vinyals.Le.2014,Cho.Van-Merrienboer.Gulcehre.ea.2014`.

Here, the encoder RNN will take a variable-length sequence as input 
and transform it into a fixed-shape hidden state.
Later, in :numref:`chap_attention-and-transformers`,
we will introduce attention mechanisms, 
which allow us to access encoded inputs
without having to compress the entire input
into a single fixed-length representation.

Then to generate the output sequence, 
one token at a time,
the decoder model, 
consisting of a separate RNN,
will predict each successive target token
given both the input sequence
and the preceding tokens in the output.
During training, the decoder will typically
be conditioned upon the preceding tokens
in the official "ground truth" label. 
However, at test time, we will want to condition
each output of the decoder on the tokens already predicted. 
Note that if we ignore the encoder,
the decoder in a sequence-to-sequence architecture 
behaves just like a normal language model.
:numref:`fig_seq2seq` illustrates
how to use two RNNs
for sequence-to-sequence learning
in machine translation.


![Sequence-to-sequence learning with an RNN encoder and an RNN decoder.](../img/seq2seq.svg)
:label:`fig_seq2seq`

In :numref:`fig_seq2seq`,
the special "&lt;eos&gt;" token
marks the end of the sequence.
Our model can stop making predictions
once this token is generated.
At the initial time step of the RNN decoder,
there are two special design decisions to be aware of:
First, we begin every input with a special 
beginning-of-sequence "&lt;bos&gt;" token.
Second, we may feed
the final hidden state of the encoder
into the decoder
at every single decoding time step :cite:`Cho.Van-Merrienboer.Gulcehre.ea.2014`.
In some other designs,
such as that of :citet:`Sutskever.Vinyals.Le.2014`,
the final hidden state of the RNN encoder
is used
to initiate the hidden state of the decoder
only at the first decoding step.


In [1]:
import collections
import math
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
%matplotlib inline

import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

## Teacher Forcing

While running the encoder on the input sequence
is relatively straightforward,
handling the input and output 
of the decoder requires more care. 
The most common approach is sometimes called *teacher forcing*.
Here, the original target sequence (token labels)
is fed into the decoder as input.
More concretely,
the special beginning-of-sequence token
and the original target sequence,
excluding the final token,
are concatenated as input to the decoder,
while the decoder output (labels for training) is
the original target sequence,
shifted by one token:
"&lt;bos&gt;", "Ils", "regardent", "." $\rightarrow$
"Ils", "regardent", ".", "&lt;eos&gt;" (:numref:`fig_seq2seq`).

Our implementation in
:numref:`subsec_loading-seq-fixed-len`
prepared training data for teacher forcing,
where shifting tokens for self-supervised learning
is similar to the training of language models in
:numref:`sec_language-model`.
An alternative approach is
to feed the *predicted* token
from the previous time step
as the current input to the decoder.


In the following, we explain the design 
depicted in :numref:`fig_seq2seq`
in greater detail.
We will train this model for machine translation
on the English--French dataset as introduced in
:numref:`sec_machine_translation`.

## Encoder

Recall that the encoder transforms an input sequence of variable length
into a fixed-shape *context variable* $\mathbf{c}$ (see :numref:`fig_seq2seq`).


Consider a single sequence example (batch size 1).
Suppose the input sequence is $x_1, \ldots, x_T$, 
such that $x_t$ is the $t^{\textrm{th}}$ token.
At time step $t$, the RNN transforms
the input feature vector $\mathbf{x}_t$ for $x_t$
and the hidden state $\mathbf{h} _{t-1}$ 
from the previous time step 
into the current hidden state $\mathbf{h}_t$.
We can use a function $f$ to express 
the transformation of the RNN's recurrent layer:

$$\mathbf{h}_t = f(\mathbf{x}_t, \mathbf{h}_{t-1}). $$

In general, the encoder transforms 
the hidden states at all time steps
into a context variable through a customized function $q$:

$$\mathbf{c} =  q(\mathbf{h}_1, \ldots, \mathbf{h}_T).$$

For example, in :numref:`fig_seq2seq`,
the context variable is just the hidden state $\mathbf{h}_T$
corresponding to the encoder RNN's representation
after processing the final token of the input sequence.

In this example, we have used a unidirectional RNN
to design the encoder,
where the hidden state only depends on the input subsequence 
at and before the time step of the hidden state.
We can also construct encoders using bidirectional RNNs.
In this case, a hidden state depends on the subsequence before and after the time step 
(including the input at the current time step), 
which encodes the information of the entire sequence.


Now let's [**implement the RNN encoder**].
Note that we use an *embedding layer*
to obtain the feature vector for each token in the input sequence.
The weight of an embedding layer is a matrix,
where the number of rows corresponds to 
the size of the input vocabulary (`vocab_size`)
and number of columns corresponds to 
the feature vector's dimension (`embed_size`).
For any input token index $i$,
the embedding layer fetches the $i^{\textrm{th}}$ row 
(starting from 0) of the weight matrix
to return its feature vector.
Here we implement the encoder with a multilayer GRU.


In [3]:
def init_seq2seq(module):  #@save
    """Initialize weights for sequence-to-sequence learning."""
    if type(module) == nn.Linear:
         nn.init.xavier_uniform_(module.weight)
    if type(module) == nn.GRU:
        for param in module._flat_weights_names:
            if "weight" in param:
                nn.init.xavier_uniform_(module._parameters[param])

In [4]:
from utils.helper import Encoder

class Seq2SeqEncoder(Encoder):  # @save
    """The RNN encoder for sequence-to-sequence learning."""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size,
            num_hiddens,
            num_layers,
            dropout=dropout if num_layers > 1 else 0
        )
        self.apply(init_seq2seq)

    def forward(self, X, *args):
        embs = self.embedding(X.t().type(torch.int64))  # (T, B, E)
        outputs, state = self.rnn(embs)
        return outputs, state


Let's use a concrete example
to [**illustrate the above encoder implementation.**]
Below, we instantiate a two-layer GRU encoder
whose number of hidden units is 16.
Given a minibatch of sequence inputs `X`
(batch size $=4$; number of time steps $=9$),
the hidden states of the final layer
at all the time steps
(`enc_outputs` returned by the encoder's recurrent layers)
are a tensor of shape
(number of time steps, batch size, number of hidden units).


In [5]:
from utils.helper import GRU
    
vocab_size, embed_size, num_hiddens, num_layers = 10, 8, 16, 2
batch_size, num_steps = 4, 9
encoder = Seq2SeqEncoder(vocab_size, embed_size, num_hiddens, num_layers)
X = torch.zeros((batch_size, num_steps))
enc_outputs, enc_state = encoder(X)

Since we are using a GRU here,
the shape of the multilayer hidden states
at the final time step is
(number of hidden layers, batch size, number of hidden units).


In [6]:
def check_shape(x, shape):
    assert tuple(x.shape) == shape, \
        f"Expected shape {shape}, got {tuple(x.shape)}"
    
check_shape(enc_state, (num_layers, batch_size, num_hiddens))

## [**Decoder**]
:label:`sec_seq2seq_decoder`

Given a target output sequence $y_1, y_2, \ldots, y_{T'}$
for each time step $t'$
(we use $t^\prime$ to differentiate from the input sequence time steps),
the decoder assigns a predicted probability
to each possible token occurring at step $y_{t'+1}$
conditioned upon the previous tokens in the target
$y_1, \ldots, y_{t'}$ 
and the context variable 
$\mathbf{c}$, i.e., $P(y_{t'+1} \mid y_1, \ldots, y_{t'}, \mathbf{c})$.

To predict the subsequent token $t^\prime+1$ in the target sequence,
the RNN decoder takes the previous step's target token $y_{t^\prime}$,
the hidden RNN state from the previous time step $\mathbf{s}_{t^\prime-1}$,
and the context variable $\mathbf{c}$ as its input,
and transforms them into the hidden state 
$\mathbf{s}_{t^\prime}$ at the current time step.
We can use a function $g$ to express 
the transformation of the decoder's hidden layer:

$$\mathbf{s}_{t^\prime} = g(y_{t^\prime-1}, \mathbf{c}, \mathbf{s}_{t^\prime-1}).$$
:eqlabel:`eq_seq2seq_s_t`

After obtaining the hidden state of the decoder,
we can use an output layer and the softmax operation 
to compute the predictive distribution
$p(y_{t^{\prime}+1} \mid y_1, \ldots, y_{t^\prime}, \mathbf{c})$ 
over the subsequent output token ${t^\prime+1}$.

Following :numref:`fig_seq2seq`,
when implementing the decoder as follows,
we directly use the hidden state at the final time step
of the encoder
to initialize the hidden state of the decoder.
This requires that the RNN encoder and the RNN decoder 
have the same number of layers and hidden units.
To further incorporate the encoded input sequence information,
the context variable is concatenated
with the decoder input at all the time steps.
To predict the probability distribution of the output token,
we use a fully connected layer
to transform the hidden state 
at the final layer of the RNN decoder.


In [7]:
from utils.helper import Decoder

class Seq2SeqDecoder(Decoder):
    """The RNN decoder for sequence-to-sequence learning."""
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers,
                 dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(
            embed_size + num_hiddens,
            num_hiddens,
            num_layers,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dense = nn.LazyLinear(vocab_size)
        self.apply(init_seq2seq)

    def init_state(self, enc_all_outputs, *args):
        # enc_all_outputs = (enc_outputs, enc_state)
        return enc_all_outputs

    def forward(self, X, state):
        # X: (batch_size, num_steps)
        embs = self.embedding(X.t().long())   # (T, B, E)

        enc_output, hidden_state = state
        context = enc_output[-1]              # (B, H)

        # Broadcast context to (T, B, H)
        context = context.unsqueeze(0).repeat(embs.shape[0], 1, 1)

        # Concatenate on feature dimension
        embs_and_context = torch.cat((embs, context), dim=-1)

        outputs, hidden_state = self.rnn(embs_and_context, hidden_state)
        outputs = self.dense(outputs).swapaxes(0, 1)

        return outputs, [enc_output, hidden_state]


To [**illustrate the implemented decoder**],
below we instantiate it with the same hyperparameters from the aforementioned encoder.
As we can see, the output shape of the decoder becomes (batch size, number of time steps, vocabulary size),
where the final dimension of the tensor stores the predicted token distribution.


In [8]:
decoder = Seq2SeqDecoder(vocab_size, embed_size, num_hiddens, num_layers)
state = decoder.init_state(encoder(X))
dec_outputs, state = decoder(X, state)

In [9]:
check_shape(dec_outputs, (batch_size, num_steps, vocab_size))
check_shape(state[1], (num_layers, batch_size, num_hiddens))

The layers in the above RNN encoder--decoder model 
are summarized in :numref:`fig_seq2seq_details`.

![Layers in an RNN encoder--decoder model.](../img/seq2seq-details.svg)
:label:`fig_seq2seq_details`



## Encoder--Decoder for Sequence-to-Sequence Learning


Putting it all together in code yields the following:


In [11]:
from utils.helper import EncoderDecoder

class Seq2Seq(EncoderDecoder):  # @save
    """The RNN encoder--decoder for sequence-to-sequence learning."""
    def __init__(self, encoder, decoder, tgt_pad, lr):
        super().__init__(encoder, decoder)
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_pad = tgt_pad
        self.lr = lr
        self.save_hyperparameters()

    def training_step(self, batch):
        # batch: (src, tgt_in, src_valid_len, tgt_out)
        Y_hat = self(*batch[:-1])          # (B, T, V)
        l = self.loss(Y_hat, batch[-1])    # masked CE
        self.plot('loss', l, train=True)
        return l


    def validation_step(self, batch):
        Y_hat = self(*batch[:-1])
        l = self.loss(Y_hat, batch[-1])
        self.plot('loss', l, train=False)
        return l 

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

    def loss(self, Y_hat, Y):
        """
        Masked cross-entropy loss.
        Y_hat: (B, T, V)
        Y:     (B, T)
        """
        import torch.nn.functional as F

        B, T, V = Y_hat.shape
        y_hat = Y_hat.reshape(-1, V)   # (B*T, V)
        y = Y.reshape(-1)              # (B*T,)

        # per-token CE
        l = F.cross_entropy(y_hat, y, reduction='none')  # (B*T,)

        # mask out <pad>
        mask = (y != self.tgt_pad).float()
        return (l * mask).sum() / mask.sum().clamp_min(1)


## Loss Function with Masking

At each time step, the decoder predicts 
a probability distribution for the output tokens.
As with language modeling, 
we can apply softmax 
to obtain the distribution
and calculate the cross-entropy loss for optimization.
Recall from :numref:`sec_machine_translation`
that the special padding tokens
are appended to the end of sequences
and so sequences of varying lengths
can be efficiently loaded
in minibatches of the same shape.
However, prediction of padding tokens
should be excluded from loss calculations.
To this end, we can 
[**mask irrelevant entries with zero values**]
so that multiplication 
of any irrelevant prediction
with zero equates to zero.


In [12]:
# from utils.helper import add_to_class

# @add_to_class(Seq2Seq)
# def loss(self, Y_hat, Y):
#     l = super(Seq2Seq, self).loss(Y_hat, Y, averaged=False)
#     mask = (Y.reshape(-1) != self.tgt_pad).type(torch.float32)
#     return (l * mask).sum() / mask.sum()

## [**Training**]
:label:`sec_seq2seq_training`

Now we can [**create and train an RNN encoder--decoder model**]
for sequence-to-sequence learning on the machine translation dataset.


In [13]:
# @add_to_class(Seq2Seq)  # @save
# def loss(self, Y_hat, Y):
#     """
#     Y_hat: (B, T, V)
#     Y:     (B, T)
#     Masks out positions where Y == tgt_pad.
#     """
#     import torch.nn.functional as F
#     B, T, V = Y_hat.shape
#     y_hat = Y_hat.reshape(-1, V)
#     y = Y.reshape(-1)

#     loss = F.cross_entropy(y_hat, y, reduction='none')
#     mask = (y != self.tgt_pad).float()
#     return (loss * mask).sum() / mask.sum().clamp_min(1.0)

In [14]:
from utils.helper import MTFraEng
from utils.helper import Trainer

data = MTFraEng(batch_size=128)
embed_size, num_hiddens, num_layers, dropout = 256, 256, 2, 0.2
encoder = Seq2SeqEncoder(
    len(data.src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqDecoder(
    len(data.tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
model = Seq2Seq(encoder, decoder, tgt_pad=data.tgt_vocab['<pad>'],lr=0.005)
trainer = Trainer(max_epochs=30, gradient_clip_val=1)
trainer.fit(model, data)


Extracting fra-eng.zip into ../data ...
epoch 1, train loss 4.029364, val loss 3.485078
epoch 2, train loss 2.777527, val loss 3.268637
epoch 3, train loss 2.326952, val loss 3.067308
epoch 4, train loss 2.025199, val loss 2.941823
epoch 5, train loss 1.766696, val loss 2.924460
epoch 6, train loss 1.529212, val loss 2.926578
epoch 7, train loss 1.357122, val loss 3.001804
epoch 8, train loss 1.174656, val loss 3.083859
epoch 9, train loss 1.028002, val loss 3.091661
epoch 10, train loss 0.893785, val loss 3.215111
epoch 11, train loss 0.784263, val loss 3.281314
epoch 12, train loss 0.679655, val loss 3.244402
epoch 13, train loss 0.590286, val loss 3.453284
epoch 14, train loss 0.512027, val loss 3.350372
epoch 15, train loss 0.444368, val loss 3.515782
epoch 16, train loss 0.396624, val loss 3.433818
epoch 17, train loss 0.352265, val loss 3.607399
epoch 18, train loss 0.326554, val loss 3.518737
epoch 19, train loss 0.305923, val loss 3.638408
epoch 20, train loss 0.276262, val los

## [**Prediction**]

To predict the output sequence
at each step, 
the predicted token from the previous
time step is fed into the decoder as an input.
One simple strategy is to sample whichever token
that has been assigned by the decoder the highest probability
when predicting at each step.
As in training, at the initial time step
the beginning-of-sequence ("&lt;bos&gt;") token
is fed into the decoder.
This prediction process
is illustrated in :numref:`fig_seq2seq_predict`.
When the end-of-sequence ("&lt;eos&gt;") token is predicted,
the prediction of the output sequence is complete.


![Predicting the output sequence token by token using an RNN encoder--decoder.](../img/seq2seq-predict.svg)
:label:`fig_seq2seq_predict`

In the next section, we will introduce 
more sophisticated strategies 
based on beam search (:numref:`sec_beam-search`).


In [16]:
from utils.helper import add_to_class

@add_to_class(EncoderDecoder)  #@save
def predict_step(self, batch, device, num_steps,
                 save_attention_weights=False):
    batch = [a.to(device) for a in batch]
    src, tgt, src_valid_len, _ = batch
    enc_all_outputs = self.encoder(src, src_valid_len)
    dec_state = self.decoder.init_state(enc_all_outputs, src_valid_len)
    outputs, attention_weights = [tgt[:, (0)].unsqueeze(1), ], []
    for _ in range(num_steps):
        Y, dec_state = self.decoder(outputs[-1], dec_state)
        outputs.append(Y.argmax(2))
        # Save attention weights (to be covered later)
        if save_attention_weights:
            attention_weights.append(self.decoder.attention_weights)
    return torch.cat(outputs[1:], 1), attention_weights

## Evaluation of Predicted Sequences

We can evaluate a predicted sequence
by comparing it with the
target sequence (the ground truth).
But what precisely is the appropriate measure 
for comparing similarity between two sequences?


Bilingual Evaluation Understudy (BLEU),
though originally proposed for evaluating
machine translation results :cite:`Papineni.Roukos.Ward.ea.2002`,
has been extensively used in measuring
the quality of output sequences for different applications.
In principle, for any $n$-gram (:numref:`subsec_markov-models-and-n-grams`) in the predicted sequence,
BLEU evaluates whether this $n$-gram appears
in the target sequence.

Denote by $p_n$ the precision of an $n$-gram,
defined as the ratio 
of the number of matched $n$-grams in
the predicted and target sequences
to the number of $n$-grams in the predicted sequence.
To explain, given a target sequence $A$, $B$, $C$, $D$, $E$, $F$,
and a predicted sequence $A$, $B$, $B$, $C$, $D$,
we have $p_1 = 4/5$,  $p_2 = 3/4$, $p_3 = 1/3$, and $p_4 = 0$.
Now let $\textrm{len}_{\textrm{label}}$ and $\textrm{len}_{\textrm{pred}}$
be the numbers of tokens in the target sequence 
and the predicted sequence, respectively.
Then, BLEU is defined as

$$ \exp\left(\min\left(0, 1 - \frac{\textrm{len}_{\textrm{label}}}{\textrm{len}_{\textrm{pred}}}\right)\right) \prod_{n=1}^k p_n^{1/2^n},$$
:eqlabel:`eq_bleu`

where $k$ is the longest $n$-gram for matching.

Based on the definition of BLEU in :eqref:`eq_bleu`,
whenever the predicted sequence is the same as the target sequence, BLEU is 1.
Moreover,
since matching longer $n$-grams is more difficult,
BLEU assigns a greater weight
when a longer $n$-gram has high precision.
Specifically, when $p_n$ is fixed,
$p_n^{1/2^n}$ increases as $n$ grows (the original paper uses $p_n^{1/n}$).
Furthermore,
since
predicting shorter sequences
tends to yield a higher $p_n$ value,
the coefficient before the multiplication term in :eqref:`eq_bleu`
penalizes shorter predicted sequences.
For example, when $k=2$,
given the target sequence $A$, $B$, $C$, $D$, $E$, $F$ and the predicted sequence $A$, $B$,
although $p_1 = p_2 = 1$, the penalty factor $\exp(1-6/2) \approx 0.14$ lowers the BLEU.

We [**implement the BLEU measure**] as follows.


In [17]:
def bleu(pred_seq, label_seq, k):  #@save
    """Compute the BLEU."""
    pred_tokens, label_tokens = pred_seq.split(' '), label_seq.split(' ')
    len_pred, len_label = len(pred_tokens), len(label_tokens)
    score = math.exp(min(0, 1 - len_label / len_pred))
    for n in range(1, min(k, len_pred) + 1):
        num_matches, label_subs = 0, collections.defaultdict(int)
        for i in range(len_label - n + 1):
            label_subs[' '.join(label_tokens[i: i + n])] += 1
        for i in range(len_pred - n + 1):
            if label_subs[' '.join(pred_tokens[i: i + n])] > 0:
                num_matches += 1
                label_subs[' '.join(pred_tokens[i: i + n])] -= 1
        score *= math.pow(num_matches / (len_pred - n + 1), math.pow(0.5, n))
    return score

In the end,
we use the trained RNN encoder--decoder
to [**translate a few English sentences into French**]
and compute the BLEU of the results.


In [18]:
import inspect
print(inspect.signature(model.predict_step))

(batch, device, num_steps, save_attention_weights=False)


In [ ]:
engs = ['go .', 'i lost .', "he's calm .", "i'm home ."]
fras = ['va !', "j'ai perdu .", 'il est calme .', 'je suis chez moi .']

batch = data.build(engs, fras)

device = torch.device("cpu")
preds, _ = model.predict_step(batch, device, data.num_steps)

for en, fr, p in zip(engs, fras, preds):
    translation = []
    for idx in p:
        tok = data.tgt_vocab.idx_to_token[int(idx)]
        if tok == '<eos>':
            break
        if tok not in ('<bos>', '<pad>'):
            translation.append(tok)
    pred_sent = " ".join(translation)
    print(f"{en} => {pred_sent}, bleu, {bleu(pred_sent, fr, k=2):.3f}")


go . => va !, bleu, 1.000
i lost . => j'ai perdu ., bleu, 1.000
he's calm . => il court ., bleu, 0.000
i'm home . => je suis chez moi ., bleu, 1.000


## Summary

Following the design of the encoder--decoder architecture, we can use two RNNs to design a model for sequence-to-sequence learning.
In encoder--decoder training, the teacher forcing approach feeds original output sequences (in contrast to predictions) into the decoder.
When implementing the encoder and the decoder, we can use multilayer RNNs.
We can use masks to filter out irrelevant computations, such as when calculating the loss.
For evaluating output sequences,
BLEU is a popular measure that matches $n$-grams between the predicted sequence and the target sequence.

